# Temporal attention for SMAD time-course classification

This experiment compares a two-branch convolutional baseline with cross-attention between absolute signal levels and first-order differences.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from tensorflow import keras

from deeplearning_examples.timecourse import load_smad_classification_data
from tensorflow_attention.model import build_attention_model, build_attention_probe, build_baseline_model
from tensorflow_attention.training import TrainingConfig, train

## 2. Data and grouped experiment split

Labels are converted to one-hot vectors only after a stratified held-out split. TensorFlow uses channels-last input.

In [ ]:
dataset = load_smad_classification_data()
x_train, x_test, y_train_int, y_test_int = train_test_split(
    dataset.channels_last,
    dataset.targets,
    test_size=0.25,
    random_state=42,
    stratify=dataset.targets,
)
y_train = keras.utils.to_categorical(y_train_int, len(dataset.class_names))
y_test = keras.utils.to_categorical(y_test_int, len(dataset.class_names))
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

## 3. Baseline model

Both branches are encoded convolutionally and pooled without an explicit attention operation.

In [ ]:
baseline_model = build_baseline_model(
    input_length=x_train.shape[1], num_classes=len(dataset.class_names)
)
baseline_model.summary()
baseline_history = train(
    baseline_model, x_train, y_train, config=TrainingConfig(epochs=200, patience=20)
)

## 4. Attention model

Cross-attention lets level-derived features query features derived from temporal differences before global pooling.

In [ ]:
attention_model = build_attention_model(
    input_length=x_train.shape[1], num_classes=len(dataset.class_names)
)
attention_model.summary()
attention_history = train(
    attention_model, x_train, y_train, config=TrainingConfig(epochs=200, patience=20)
)

## 5. Compare optimization behavior

Validation accuracy measures whether the additional attention operation improves generalization rather than only training fit.

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(baseline_history.history["val_categorical_accuracy"], label="baseline")
plt.plot(attention_history.history["val_categorical_accuracy"], label="attention")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.legend()
plt.tight_layout()

## 6. Held-out evaluation

Class-wise metrics distinguish a general improvement from a gain restricted to one stimulation condition.

In [ ]:
for name, model in {"baseline": baseline_model, "attention": attention_model}.items():
    predicted = model.predict(x_test, verbose=0).argmax(1)
    print(f"\n{name}")
    print(classification_report(y_test_int, predicted, target_names=dataset.class_names))

predicted = attention_model.predict(x_test, verbose=0).argmax(1)
ConfusionMatrixDisplay.from_predictions(
    y_test_int, predicted, display_labels=dataset.class_names, xticks_rotation=25
)
plt.tight_layout()

## 7. Inspect the attended representation

The probe exposes the post-attention feature tensor. This is useful for comparing temporal structure, but it should not be interpreted as a causal attribution map.

In [ ]:
probe = build_attention_probe(attention_model)
representation = probe.predict(x_test[:12], verbose=0)
plt.figure(figsize=(9, 4))
plt.imshow(representation[0].T, aspect="auto", cmap="viridis")
plt.xlabel("encoded time")
plt.ylabel("feature channel")
plt.colorbar(label="activation")
plt.tight_layout()

## 8. Interpretation

A fair conclusion should report repeated splits or cross-validation. With highly correlated single-cell measurements, the experimental unit used for splitting is as important as the architecture.